In [2]:
!nvidia-smi

Thu Oct  8 07:05:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from numba import cuda
import os

device = cuda.get_current_device()

print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


In [6]:
import numpy as np
import time
from numba import cuda

# 1. CPU Reference Implementation
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)

    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [7]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)

    h_in = np.random.rand(N).astype(np.float32)

    # CPU
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    # GPU setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU total time
    t0 = time.perf_counter()

    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)

    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)

    h_out = d_out.copy_to_host()

    cuda.synchronize()

    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU compute only
    t0 = time.perf_counter()

    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)

    cuda.synchronize()

    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [8]:
sizes = [100, 1000, 10000, 100000, 1000000, 5000000, 10000000]

for N in sizes:
    cpu, gpu_total, gpu_compute = benchmark_run(N)

    winner = "GPU" if gpu_total < cpu else "CPU"

    print(
        f"N={N:,} | "
        f"CPU={cpu:.3f} ms | "
        f"GPU Total={gpu_total:.3f} ms | "
        f"GPU Kernel={gpu_compute:.3f} ms | "
        f"Winner={winner}"
    )

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


N=100 | CPU=0.077 ms | GPU Total=8.467 ms | GPU Kernel=0.125 ms | Winner=CPU
N=1,000 | CPU=0.510 ms | GPU Total=0.886 ms | GPU Kernel=0.112 ms | Winner=CPU
N=10,000 | CPU=33.956 ms | GPU Total=0.767 ms | GPU Kernel=8.158 ms | Winner=GPU
N=100,000 | CPU=107.239 ms | GPU Total=8.063 ms | GPU Kernel=0.119 ms | Winner=GPU
N=1,000,000 | CPU=810.834 ms | GPU Total=5.432 ms | GPU Kernel=0.145 ms | Winner=GPU
N=5,000,000 | CPU=nan ms | GPU Total=13.777 ms | GPU Kernel=0.296 ms | Winner=CPU
N=10,000,000 | CPU=nan ms | GPU Total=28.661 ms | GPU Kernel=0.519 ms | Winner=CPU


In [9]:
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0


N = 1000

h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")

defective_kernel[blocks_per_grid, threads_per_block](d_arr)

cuda.synchronize()

print("Execution finished (or did it crash?). Checking host copy...")

h_result = d_arr.copy_to_host()

print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


nvidia-smi


In [10]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0


N = 37
threads_per_block = 8

blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

log_b = cuda.device_array(
    blocks_per_grid * threads_per_block,
    dtype=np.int32
)

log_t = cuda.device_array(
    blocks_per_grid * threads_per_block,
    dtype=np.int32
)

log_g = cuda.device_array(
    blocks_per_grid * threads_per_block,
    dtype=np.int32
)

log_a = cuda.device_array(
    blocks_per_grid * threads_per_block,
    dtype=np.int32
)

inspect_threads_kernel[blocks_per_grid, threads_per_block](
    log_b, log_t, log_g, log_a, N
)

cuda.synchronize()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)

for i in range(32, 40):
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(
        f"{log_g[i]:<10} | "
        f"{log_b[i]:<10} | "
        f"{log_t[i]:<10} | "
        f"{status:<10}"
    )

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [11]:
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)

    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]


@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)

    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

In [12]:
N = 50000
h_arr = np.ones(N, dtype=np.float32)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

for trial in range(1, 6):

    d_arr = cuda.to_device(h_arr)
    d_total = cuda.to_device(np.array([0.0], dtype=np.float32))

    naive_sum_kernel[blocks_per_grid, threads_per_block](
        d_arr, d_total
    )

    cuda.synchronize()

    result = d_total.copy_to_host()[0]
    deviation = 50000.0 - result

    print(
        f"Trial {trial}: "
        f"Target=50000.0 | "
        f"Naive={result} | "
        f"Lost Updates={deviation}"
    )

Trial 1: Target=50000.0 | Naive=2.0 | Lost Updates=49998.0
Trial 2: Target=50000.0 | Naive=2.0 | Lost Updates=49998.0
Trial 3: Target=50000.0 | Naive=2.0 | Lost Updates=49998.0
Trial 4: Target=50000.0 | Naive=2.0 | Lost Updates=49998.0
Trial 5: Target=50000.0 | Naive=2.0 | Lost Updates=49998.0


In [13]:
import math

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):

    x, y = cuda.grid(2)

    if x < width and y < height:

        dx = float(x - center_x)
        dy = float(y - center_y)

        dist = math.sqrt(dx * dx + dy * dy)

        max_dist = math.sqrt(
            float(center_x**2 + center_y**2)
        )

        intensity = 1.0 - (dist / max_dist)

        if intensity < 0.0:
            intensity = 0.0

        d_canvas[y, x] = intensity


WIDTH, HEIGHT = 1024, 1024

canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)

d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)

blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[
    blocks_2d,
    threads_2d
](
    d_canvas,
    WIDTH,
    HEIGHT,
    WIDTH // 2,
    HEIGHT // 2
)

cuda.synchronize()

result_canvas = d_canvas.copy_to_host()

print("Grid dimensions:", blocks_2d)
print("Generated shape:", result_canvas.shape)
print("Center value:", result_canvas[512, 512])
print("Corner value:", result_canvas[0, 0])

Grid dimensions: (64, 64)
Generated shape: (1024, 1024)
Center value: 1.0
Corner value: 0.0


In [17]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):

    h = hashlib.sha256()

    h.update(student_id_str.strip().encode('utf-8'))

    h.update(cuda.get_current_device().name)

    h.update(
        d_canvas.copy_to_host()[:10, :10].tobytes()
    )

    h.update(
        str(exp_table_crossover).encode('utf-8')
    )

    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)


In [18]:
import hashlib
from numba import cuda

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()

    h.update(student_id_str.strip().encode('utf-8'))
    h.update(cuda.get_current_device().name.encode('utf-8')) # Добавлен .encode('utf-8')
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))

    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)

    return digest

STUDENT_ID = "230103223"
CROSSOVER_N = 4096

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: B87F1CB9602E9744


'B87F1CB9602E9744'